# **V) Data Transform**

Merges, and transforms raw economic and Google Trends data into a clean, stationary, and model-ready weekly dataset.

1. [**Import Data**](#1-import--data)
2. [**Renaming**](#2-renaming)
3. [**Stationarity Transformations**](#3-transformations)
4. [**Stationarity Testing (ADF Test)**](#4-stationarity)

### **1) Import Data** <a id="1-import--data"></a>

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

btc_raw = yf.download("BTC-USD", start="2010-01-01", end="2025-12-31", interval="1d")
btc_w = btc_raw['Close'].resample('W').last() 
btc_w.name = "Bitcoin_Price"

/var/folders/bs/5f0ys9r13_l1yqx_3qv0ncg80000gn/T/ipykernel_98075/3411794568.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc_raw = yf.download("BTC-USD", start="2010-01-01", end="2025-12-31", interval="1d")
[*********************100%***********************]  1 of 1 completed


In [2]:
# Bloomberg: Shift Friday to Sunday
bbg = pd.read_csv('data/raw/Bloomberg.csv', parse_dates=['DATES'], index_col='DATES')
bbg.index = bbg.index + pd.Timedelta(days=2)

# Exchange Rate: Using specific column indices (8 and 12)
eer = pd.read_csv(
    'data/raw/Effective Exchange Rate.csv',
    skiprows=13, # Start exactly where data begins
    header=None,
    usecols=[8, 12],
    names=['date', 'EER'],
    parse_dates=['date'],
    dayfirst=True
)
eer['EER'] = pd.to_numeric(eer['EER'], errors='coerce')
eer_w = eer.set_index('date').resample('W').mean()

# EMV
emv = pd.read_csv('data/raw/All_Infectious_EMV_Data.csv')
emv.index = pd.to_datetime(emv[['year', 'month', 'day']])
emv_w = emv[['daily_infect_emv_index']].resample('W').mean()

# Policy Risk
pol = pd.read_csv('data/raw/US Policy Risk Index.csv')
pol.index = pd.to_datetime(pol[['year', 'month', 'day']])
pol_w = pol[['daily_policy_index']].resample('W').mean()

# GPRD
gpr = pd.read_csv('data/raw/GPRD.csv')
gpr.index = pd.to_datetime(gpr['DAY'].astype(str), format='%Y%m%d')
gpr['GPRD'] = pd.to_numeric(gpr['GPRD'], errors='coerce')
gpr_w = gpr[['GPRD']].resample('W').mean()

# News Sentiment
nws = pd.read_csv('data/raw/US_SF_News_Sentiment.csv', parse_dates=['date'], index_col='date')
nws_w = nws.resample('W').mean()

# Google Indexes (Load and Rename index_0_100)
def load_gt(path, name):
    df = pd.read_csv(path, parse_dates=['date'], index_col='date')
    return df[['index_0_100']].rename(columns={'index_0_100': name})

# MPE Index
mpe = pd.read_csv('data/weekly_mpe_index.csv', parse_dates=['date'], index_col='date')
# Remove timezone (UTC) to match naive dates in other files
mpe.index = mpe.index.tz_localize(None)

gt_climate = load_gt('data/google_climate_change_2014_2025.csv', 'google_climate')
gt_inflation = load_gt('data/google_inflation_2014_2025.csv', 'google_inflation')
gt_recession = load_gt('data/google_recession_2014_2025.csv', 'google_recession')


final = pd.concat([
    bbg, emv_w, pol_w, gpr_w, nws_w, eer_w, 
    gt_climate, gt_inflation, gt_recession, mpe, btc_w
], axis=1).sort_index().dropna()

final.index.name = 'date'

final

/var/folders/bs/5f0ys9r13_l1yqx_3qv0ncg80000gn/T/ipykernel_98075/675680800.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nws = pd.read_csv('data/raw/US_SF_News_Sentiment.csv', parse_dates=['date'], index_col='date')


,SP500,Brent_Oil,Gold,VIX,DXY,Fred 5y Inflation Expectation,US Jobless Claims,Jobless Claims Rate,BBG US HY Return Unhedged,Fed Funds,daily_infect_emv_index,daily_policy_index,GPRD,News Sentiment,EER,google_climate,google_inflation,google_recession,mpe_index,BTC-USD
date,,,,,,,,,,,,,,,,,,,,
2014-09-21,2010.40,98.39,1215.70,12.11,84.74,2.50,295.0,1.9,1657.90,0.090,0.740000,59.440000,119.577143,0.097143,82.374,3.519661,11.290356,4.382154,-0.006073,398.821014
2014-09-28,1982.85,97.00,1218.38,14.85,85.64,2.35,290.0,1.8,1635.19,0.090,0.792857,67.532857,122.061429,0.107143,82.734,1.574585,13.744781,4.604975,0.032847,377.181000
2014-10-05,1967.90,92.31,1191.42,14.55,86.69,2.30,293.0,1.8,1646.47,0.084,1.117143,59.088571,85.725714,0.110000,83.460,1.343029,12.272126,5.050618,0.040000,320.510010
2014-10-12,1906.13,90.21,1223.16,21.24,85.91,2.40,281.0,1.8,1633.62,0.088,1.134286,73.308571,78.048571,0.121429,83.318,1.343029,13.253896,5.124892,0.036765,378.549011
2014-10-19,1886.76,86.16,1238.32,21.99,85.11,2.41,290.0,1.8,1639.95,0.090,8.691429,64.652857,72.038571,0.038571,83.294,1.389340,13.908410,5.124892,0.036932,389.545990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-02,6040.53,76.76,2798.41,16.43,108.37,2.30,222.0,1.2,2719.80,4.330,9.107143,303.452857,118.315714,0.078571,110.216,3.006696,30.695334,9.142140,-0.051813,97688.976562
2025-02-09,6025.99,74.66,2861.07,16.54,108.04,2.25,215.0,1.2,2719.67,4.330,5.397143,406.408571,117.505714,0.041429,110.474,2.886429,37.696024,7.836120,-0.009174,96500.093750
2025-02-16,6114.63,74.74,2882.53,14.77,106.71,2.24,224.0,1.2,2726.80,4.330,6.271429,266.524286,117.365714,0.034286,110.244,2.846339,33.387907,9.142140,-0.275000,96175.031250


### **2) Renaming** <a id="2-renaming"></a>

In [3]:
# Column name mapping (raw CSV -> model variable names)
column_mapping = {
    # Economic activity
    "SP500": "sp500",
    "US Jobless Claims": "claims",
    # Household consumption
    "Real Disposable Personal Income": "rdpi",
    # Firm investment
    "Brent_Oil": "brent",
    "Gold": "gold",
    "BTC-USD": "btc",
    "BBG US HY Return Unhedged": "hy",
    "GPRD": "gpr",
    "daily_infect_emv_index": "emv",
    # Monetary policy
    "Fed Funds": "ffr",
    "Fred 5y Inflation Expectation": "infl5y",
    "EER": "eer",
    "mpe_index": "mp_exp",
    # Forward-looking expectations
    "VIX": "vix",
    "DXY": "dxy",
    "News Sentiment": "news_sent",
    "daily_policy_index": "policy_risk",
    "google_climate": "gt_climate",
    "google_inflation": "gt_infl",
    "google_recession": "gt_recess",
    }


# Apply mapping
df = final.rename(columns=column_mapping)

df.head()

,sp500,brent,gold,vix,dxy,infl5y,claims,Jobless Claims Rate,hy,ffr,emv,policy_risk,gpr,news_sent,eer,gt_climate,gt_infl,gt_recess,mp_exp,btc
date,,,,,,,,,,,,,,,,,,,,
2014-09-21,2010.40,98.39,1215.70,12.11,84.74,2.50,295.0,1.9,1657.90,0.090,0.740000,59.440000,119.577143,0.097143,82.374,3.519661,11.290356,4.382154,-0.006073,398.821014
2014-09-28,1982.85,97.00,1218.38,14.85,85.64,2.35,290.0,1.8,1635.19,0.090,0.792857,67.532857,122.061429,0.107143,82.734,1.574585,13.744781,4.604975,0.032847,377.181000
2014-10-05,1967.90,92.31,1191.42,14.55,86.69,2.30,293.0,1.8,1646.47,0.084,1.117143,59.088571,85.725714,0.110000,83.460,1.343029,12.272126,5.050618,0.040000,320.510010
2014-10-12,1906.13,90.21,1223.16,21.24,85.91,2.40,281.0,1.8,1633.62,0.088,1.134286,73.308571,78.048571,0.121429,83.318,1.343029,13.253896,5.124892,0.036765,378.549011
2014-10-19,1886.76,86.16,1238.32,21.99,85.11,2.41,290.0,1.8,1639.95,0.090,8.691429,64.652857,72.038571,0.038571,83.294,1.389340,13.908410,5.124892,0.036932,389.545990


### **3) Stationarity Transformations** <a id="3-transformations"></a>

In [4]:
def log_return(x: pd.Series) -> pd.Series:
    # Safe for positive price/index levels (log difference)
    return np.log(x).diff()

def growth_rate(x: pd.Series) -> pd.Series:
    # (x_t / x_{t-1}) - 1 (percentage change)
    return x.pct_change(fill_method=None)

def log_level(x: pd.Series) -> pd.Series:
    # Natural log
    return np.log(x)

def difference(x: pd.Series) -> pd.Series:
    # First difference (level change: x_t - x_{t-1})
    return x.diff()

# Transformations

# Economic Activity
df["sp500_ret"] = log_return(df["sp500"])
df["claims_dlog"] = log_return(df["claims"]) # Log difference
# wei = absolute level
# bbk_rgdp = absolute level
# claims_rate = absolute level
# cci = absolute level

# Household Consumption Decisions
# gt_infl = absolute level
# gt_recess = absolute level

# Firm Investment Decisions
df["brent_gr"] = growth_rate(df["brent"])
df["gold_ret"] = log_return(df["gold"])
df["btc_ret"] = log_return(df["btc"])
df["hy_ret"] = growth_rate(df["hy"])
df["gpr_gr"] = growth_rate(df["gpr"])
# emv = absolute level

# Monetary Policy Decisions
df["eer_dlog"] = log_return(df["eer"])      # Log difference
df["ffr_d"] = difference(df["ffr"])         # First difference
df["ffr_d2"] = difference(df["ffr_d"])      # 2nd difference
df["infl5y_d"] = difference(df["infl5y"])   # First difference
df["mp_exp_d"] = df["mp_exp"].diff()        # First difference

# Forward-Looking Market Expectations
df["vix_gr"] = growth_rate(df["vix"])
df["dxy_gr"] = growth_rate(df["dxy"])
df["policy_risk_gr"] = growth_rate(df["policy_risk"])
# gt_climate = absolute level
# news_sent = absolute level

df["gt_infl_d"] = df["gt_infl"].diff()

df.to_csv("data/All_Data_Weekly_transformed.csv")

df.head()

,sp500,brent,gold,vix,dxy,infl5y,claims,Jobless Claims Rate,hy,ffr,...,gpr_gr,eer_dlog,ffr_d,ffr_d2,infl5y_d,mp_exp_d,vix_gr,dxy_gr,policy_risk_gr,gt_infl_d
date,,,,,,,,,,,,,,,,,,,,,
2014-09-21,2010.40,98.39,1215.70,12.11,84.74,2.50,295.0,1.9,1657.90,0.090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-09-28,1982.85,97.00,1218.38,14.85,85.64,2.35,290.0,1.8,1635.19,0.090,...,0.020776,0.004361,0.000,NaN,-0.15,0.038920,0.226259,0.010621,0.136152,2.454425
2014-10-05,1967.90,92.31,1191.42,14.55,86.69,2.30,293.0,1.8,1646.47,0.084,...,-0.297684,0.008737,-0.006,-0.006,-0.05,0.007153,-0.020202,0.012261,-0.125040,-1.472655
2014-10-12,1906.13,90.21,1223.16,21.24,85.91,2.40,281.0,1.8,1633.62,0.088,...,-0.089555,-0.001703,0.004,0.010,0.10,-0.003235,0.459794,-0.008998,0.240656,0.981770
2014-10-19,1886.76,86.16,1238.32,21.99,85.11,2.41,290.0,1.8,1639.95,0.090,...,-0.077003,-0.000288,0.002,-0.002,0.01,0.000167,0.035311,-0.009312,-0.118072,0.654513


### **4) Stationarity Testing (ADF Test)** <a id="4-stationarity"></a>

We verify if the transformed variables are stationary

In [5]:
from statsmodels.tsa.stattools import adfuller

predictors = [
    "btc_ret", "mp_exp", "news_sent", "policy_risk_gr",
    "sp500_ret", "brent_gr", "gold_ret", "hy_ret",
    "gpr_gr", "vix_gr", "dxy_gr", "emv",
    "claims_dlog", "eer_dlog", "ffr", "infl5y",
    "gt_infl", "gt_recess", "gt_climate"
]

print("Before applying diff")
results = []
for var in predictors:
    # Drop NaNs created by diff/pct_change before testing
    series = df[var].dropna()
    p_val = adfuller(series)[1]
    results.append({"Variable": var, "p-value": round(p_val, 4), "Stationary": p_val < 0.05})

stationarity_df = pd.DataFrame(results)
stationarity_df = stationarity_df.set_index("Variable")
display(stationarity_df)

print("After applying diff ")

stationary_predictors = [
    "btc_ret", "mp_exp_d", "news_sent", "policy_risk_gr",
    "sp500_ret", "brent_gr", "gold_ret", "hy_ret",
    "gpr_gr", "vix_gr", "dxy_gr", "emv",
    "claims_dlog", "eer_dlog", "ffr_d2", "infl5y_d",
    "gt_infl_d", "gt_recess", "gt_climate"
]

results = []
for var in stationary_predictors:
    # Drop NaNs created by diff/pct_change before testing
    series = df[var].dropna()
    p_val = adfuller(series)[1]
    results.append({"Variable": var, "p-value": round(p_val, 4), "Stationary": p_val < 0.05})

stationarity_df = pd.DataFrame(results)
stationarity_df = stationarity_df.set_index("Variable")
display(stationarity_df)


Before applying diff


,p-value,Stationary
Variable,,
btc_ret,0.0000,True
mp_exp,0.1425,False
news_sent,0.0089,True
policy_risk_gr,0.0000,True
sp500_ret,0.0000,True
brent_gr,0.0000,True
gold_ret,0.0000,True
hy_ret,0.0000,True
gpr_gr,0.0000,True


After applying diff 


,p-value,Stationary
Variable,,
btc_ret,0.0000,True
mp_exp_d,0.0000,True
news_sent,0.0089,True
policy_risk_gr,0.0000,True
sp500_ret,0.0000,True
brent_gr,0.0000,True
gold_ret,0.0000,True
hy_ret,0.0000,True
gpr_gr,0.0000,True
